# Standalone MLE-STAR benchmark runner

This notebook runs paired offline OOF comparisons. It never submits by default. The catalog lists ten historical Kaggle tasks; seven now have executable training adapters (one tabular, six image-classification, sharing a common timm fine-tuning pipeline). The remaining three -- object detection, segmentation, and image denoising -- are registered in the catalog but not yet implemented, and mlestar compare fails loudly rather than fabricating a result for them.

In [ ]:
REPOSITORY = 'https://github.com/Isso-W/Jiaozi.git'
BRANCH = 'codex/mlestar-kaggle-benchmarks'
!rm -rf /content/mlestar-dataops
!git clone --depth 1 --branch {BRANCH} {REPOSITORY} /content/mlestar-dataops
%cd /content/mlestar-dataops
!python -m pip install -q '.[vision,llm,kaggle,dev]'
!python -m mlestar.cli benchmarks

In [ ]:
# Optional: add exactly one KAGGLE_API_TOKEN secret. This cell never prints it.
import os
try:
    from google.colab import userdata
    token = userdata.get('KAGGLE_API_TOKEN')
    if token:
        os.environ['KAGGLE_API_TOKEN'] = token
except ImportError:
    pass

# Accept each competition's rules in the Kaggle UI before downloading its data.
SUBMIT = False

In [ ]:
# Reusable Kaggle competition fetcher, used by every task cell below.
# Uses the KAGGLE_API_TOKEN secret from the previous cell -- the kaggle CLI
# reads it directly from the environment, no credentials file needed.
# Accept each competition's rules on kaggle.com before running its cell, or
# upload that task's files to its DATA_ROOT yourself and skip the fetch.
import os
import pathlib
import zipfile


def fetch_kaggle_competition(slug: str, data_root: str, marker_file: str) -> None:
    root = pathlib.Path(data_root)
    root.mkdir(parents=True, exist_ok=True)
    if (root / marker_file).exists():
        return
    if not os.environ.get('KAGGLE_API_TOKEN'):
        raise RuntimeError(
            f'No {marker_file} in {data_root} and no KAGGLE_API_TOKEN secret set. '
            'Either set the secret in the previous cell or upload the data yourself.'
        )
    !kaggle competitions download -c {slug} -p {data_root}
    outer_zip = root / f'{slug}.zip'
    if outer_zip.exists():
        with zipfile.ZipFile(outer_zip) as archive:
            archive.extractall(root)
    # Some competitions nest further archives one level deeper inside the
    # outer zip: leaf-classification splits train/test/sample_submission
    # into their own *.csv.zip archives, while dogs-vs-cats-redux-kernels-
    # edition nests plain train.zip/test.zip image archives. Extract every
    # nested zip (other than the outer one, already handled above) so both
    # patterns produce their expected on-disk files.
    for nested_zip in root.glob('*.zip'):
        if nested_zip == outer_zip:
            continue
        with zipfile.ZipFile(nested_zip) as archive:
            archive.extractall(root)
    if not (root / marker_file).exists():
        raise RuntimeError(
            f'Download did not produce {marker_file} in {data_root}. Scroll up to the '
            'kaggle output above for the real error -- commonly the competition rules '
            f'are not accepted yet at https://www.kaggle.com/c/{slug}/rules, or '
            'KAGGLE_API_TOKEN is invalid/expired.'
        )


In [ ]:
# Leaf Classification.
DATA_ROOT = '/content/leaf-classification'
fetch_kaggle_competition('leaf-classification', DATA_ROOT, 'train.csv')
RUN_ROOT = '/content/mlestar-runs/leaf-classification'
!python -m mlestar.cli compare --benchmark leaf_classification --data-root {DATA_ROOT} --run-root {RUN_ROOT} --seeds 13 29 47 --no-submit

import pandas as pd
pd.read_csv(f'{RUN_ROOT}/comparison.csv')


In [ ]:
# Plant Pathology 2020.
DATA_ROOT = '/content/plant-pathology-2020-fgvc7'
fetch_kaggle_competition('plant-pathology-2020-fgvc7', DATA_ROOT, 'train.csv')
RUN_ROOT = '/content/mlestar-runs/plant_pathology_2020'
!python -m mlestar.cli compare --benchmark plant_pathology_2020 --data-root {DATA_ROOT} --run-root {RUN_ROOT} --seeds 13 29 47 --no-submit

import pandas as pd
pd.read_csv(f'/content/mlestar-runs/plant_pathology_2020/comparison.csv')


In [ ]:
# APTOS 2019 Blindness Detection.
DATA_ROOT = '/content/aptos2019-blindness-detection'
fetch_kaggle_competition('aptos2019-blindness-detection', DATA_ROOT, 'train.csv')
RUN_ROOT = '/content/mlestar-runs/aptos_2019'
!python -m mlestar.cli compare --benchmark aptos_2019 --data-root {DATA_ROOT} --run-root {RUN_ROOT} --seeds 13 29 47 --no-submit

import pandas as pd
pd.read_csv(f'/content/mlestar-runs/aptos_2019/comparison.csv')


In [ ]:
# Dog Breed Identification.
DATA_ROOT = '/content/dog-breed-identification'
fetch_kaggle_competition('dog-breed-identification', DATA_ROOT, 'labels.csv')
RUN_ROOT = '/content/mlestar-runs/dog_breed'
!python -m mlestar.cli compare --benchmark dog_breed --data-root {DATA_ROOT} --run-root {RUN_ROOT} --seeds 13 29 47 --no-submit

import pandas as pd
pd.read_csv(f'/content/mlestar-runs/dog_breed/comparison.csv')


In [ ]:
# Aerial Cactus Identification.
DATA_ROOT = '/content/aerial-cactus-identification'
fetch_kaggle_competition('aerial-cactus-identification', DATA_ROOT, 'train.csv')
RUN_ROOT = '/content/mlestar-runs/aerial_cactus'
!python -m mlestar.cli compare --benchmark aerial_cactus --data-root {DATA_ROOT} --run-root {RUN_ROOT} --seeds 13 29 47 --no-submit

import pandas as pd
pd.read_csv(f'/content/mlestar-runs/aerial_cactus/comparison.csv')


In [ ]:
# Dogs vs. Cats Redux.
DATA_ROOT = '/content/dogs-vs-cats-redux-kernels-edition'
fetch_kaggle_competition('dogs-vs-cats-redux-kernels-edition', DATA_ROOT, 'train')
RUN_ROOT = '/content/mlestar-runs/dogs_vs_cats'
!python -m mlestar.cli compare --benchmark dogs_vs_cats --data-root {DATA_ROOT} --run-root {RUN_ROOT} --seeds 13 29 47 --no-submit

import pandas as pd
pd.read_csv(f'/content/mlestar-runs/dogs_vs_cats/comparison.csv')


In [ ]:
# Histopathologic Cancer Detection.
DATA_ROOT = '/content/histopathologic-cancer-detection'
fetch_kaggle_competition('histopathologic-cancer-detection', DATA_ROOT, 'train_labels.csv')
RUN_ROOT = '/content/mlestar-runs/histopathologic_cancer'
!python -m mlestar.cli compare --benchmark histopathologic_cancer --data-root {DATA_ROOT} --run-root {RUN_ROOT} --seeds 13 29 47 --no-submit

import pandas as pd
pd.read_csv(f'/content/mlestar-runs/histopathologic_cancer/comparison.csv')


A public leaderboard score is not a replacement for OOF comparison. The listed competitions are historical and expected to reject new submissions; record Kaggle's actual response if and only if you explicitly set `SUBMIT = True` after reviewing the competition rules and local submission schema.